In [3]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu

import os
import pickle
import faiss
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from tqdm import tqdm

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
DATA="/content/drive/MyDrive/FIFI_Research/data"

MODEL_PATH="/content/drive/MyDrive/FIFI_Research/models"

val_df=pd.read_csv(
    os.path.join(DATA,"val.tsv"),
    sep="\t"
)

train_df=pd.read_csv(
    os.path.join(DATA,"train.tsv"),
    sep="\t"
)

print(val_df.shape)

(18000, 4)


In [5]:
with open(
    os.path.join(MODEL_PATH,"candidate_titles.pkl"),
    "rb"
) as f:

    candidate_titles=pickle.load(f)

index=faiss.read_index(
    os.path.join(MODEL_PATH,"faiss.index")
)

model=SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
query_embeddings=model.encode(

    val_df["generated_title"].tolist(),

    normalize_embeddings=True,

    batch_size=64,

    show_progress_bar=True
)

Batches:   0%|          | 0/282 [00:00<?, ?it/s]

In [7]:
scores,indices=index.search(

    query_embeddings,

    5
)

In [8]:
predictions=[]

for row in indices:

    predictions.append(
        candidate_titles[row[0]]
    )

val_df["prediction"]=predictions

In [9]:
from collections import Counter
import numpy as np

def token_f1(pred,gt):

    pred=pred.lower().split()

    gt=gt.lower().split()

    common=Counter(pred)&Counter(gt)

    overlap=sum(common.values())

    if overlap==0:

        return 0

    precision=overlap/len(pred)

    recall=overlap/len(gt)

    return 2*precision*recall/(precision+recall)

scores=[]

for pred,gt in zip(

    val_df["prediction"],

    val_df["original_title"]

):

    scores.append(

        token_f1(pred,gt)

    )

print("Mean Token F1 =",np.mean(scores))

Mean Token F1 = 0.7955945998016477


In [10]:
for style in [

    "technical",

    "accessible",

    "catchy"

]:

    subset=val_df[

        val_df["category"]==style

    ]

    style_scores=[]

    for pred,gt in zip(

        subset["prediction"],

        subset["original_title"]

    ):

        style_scores.append(

            token_f1(pred,gt)

        )

    print(style,np.mean(style_scores))

technical 0.9373476850056627
accessible 0.6857182255129673
catchy 0.7637178888863129


In [11]:
submission=val_df[

[
"id",
"category",
"generated_title"

]

].copy()

submission["original_title"]=val_df["prediction"]

submission.to_csv(

"/content/drive/MyDrive/FIFI_Research/submissions/FutureMinds_task2_run3.tsv",

sep="\t",

index=False

)

print(submission.head())

   id    category                                    generated_title  \
0   0   technical  Fully Convolutional Joint Detection and Regres...   
1   1      catchy  When Machines Argue: Teaching AI to Spot Fake ...   
2   2   technical  Optimized 2D Manifold Folding and Attribute Ma...   
3   3      catchy  From Snapshot to Sawdust: Rebuilding Wooden Ob...   
4   4  accessible  A Survey of How Deep Learning Improves Image, ...   

                                      original_title  
0  Preterm infants' limb-pose estimation from dep...  
1  Towards Automated Factchecking: Developing an ...  
2  Folding-based compression of point cloud attri...  
3  Fabrication-Aware Reverse Engineering for Carp...  
4                 Super-Resolution via Deep Learning  


In [12]:
submission.shape

(18000, 4)

In [13]:
submission.head()

,id,category,generated_title,original_title
0,0,technical,Fully Convolutional Joint Detection and Regres...,Preterm infants' limb-pose estimation from dep...
1,1,catchy,When Machines Argue: Teaching AI to Spot Fake ...,Towards Automated Factchecking: Developing an ...
2,2,technical,Optimized 2D Manifold Folding and Attribute Ma...,Folding-based compression of point cloud attri...
3,3,catchy,From Snapshot to Sawdust: Rebuilding Wooden Ob...,Fabrication-Aware Reverse Engineering for Carp...
4,4,accessible,"A Survey of How Deep Learning Improves Image, ...",Super-Resolution via Deep Learning
